In [ ]:
# Cell 1: Detect Kaggle / Colab GPU environment & install lightweight dependencies
import json, os, pathlib, sys, time, zipfile, subprocess

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
OUT_DIR = pathlib.Path("/kaggle/working/outputs" if IS_KAGGLE else "/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Environment: {'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')} | Out: {OUT_DIR}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers", "transformers", "accelerate", "safetensors", "Pillow", "imageio[ffmpeg]"])
import torch
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 2: Embedded Job Specification (generated by cloud-gpu-video MCP)
JOB_SPEC = json.loads(r'''{
  "job_id": "ep01_30_days_funded_survival",
  "created_at": "2026-11-26 19:02:53",
  "engine": "sdxl_lofi",
  "aspect_ratio": "9:16",
  "seed": 42,
  "character_lock": "Minimalist 2D dark survival animation illustration, consistent white round-headed stickman protagonist wearing a bright amber-gold hoodie (#F5B40B), dark slate pants and white sneakers, crisp 3px black outlines, minimalist calm eyes, deep high-contrast dark background, zero text.",
  "negative_prompt": "text, watermark, logo, blurry, deformed, extra limbs",
  "voiceover_script": "You start a fifty thousand dollar funded challenge with one rule: lose five percent in a single day, and the account dies instantly. Day one. Your inventory is simple: fifty thousand capital, two thousand five hundred daily limit, and half a percent risk per trade. Day three. Two clean winners push you up fifteen hundred. Overconfidence whispers to double your size. Day seven. Three red candles hit back to back. A massive drawdown wall towers over you. Ninety percent of traders revenge trade here and blow up. You lock the terminal instead. Day thirty. Boring half percent execution clears eight percent profit. Phase one survived. Tomorrow, Phase two begins with zero room for ego. Follow for Episode two.",
  "caption": "30 Days To Survive A $50K Funded Challenge (Episode 1). Would you survive Day 7 without revenge trading? Follow for Episode 2.",
  "hashtags": [
    "#propfirm",
    "#fundedtrader",
    "#tradingpsychology",
    "#daytrading"
  ],
  "scenes": [
    {
      "scene_id": "scene_01",
      "prompt": "Wide vertical shot of the amber-gold hoodie stickman standing calmly in a dark brutalist trading room staring at glowing holographic numbers 50K and a red 5 percent floor line.",
      "duration_sec": 6.0
    },
    {
      "scene_id": "scene_02",
      "prompt": "The amber-gold hoodie stickman walking confidently along glowing emerald green ascending bars in a dark void, soft green rim light.",
      "duration_sec": 6.0
    },
    {
      "scene_id": "scene_03",
      "prompt": "Dramatic scale contrast: tiny amber-gold hoodie stickman standing calmly at the bottom facing three towering colossal crimson red candlesticks looming like skyscrapers in a dark stormy hall.",
      "duration_sec": 6.0
    },
    {
      "scene_id": "scene_04",
      "prompt": "The amber-gold hoodie stickman calmly climbing a clean emerald green staircase out of the red storm toward warm morning light.",
      "duration_sec": 6.0
    },
    {
      "scene_id": "scene_05",
      "prompt": "The amber-gold hoodie stickman standing at the summit overlooking a glowing emerald cityscape at sunrise, peaceful victorious expression, open horizon ahead.",
      "duration_sec": 6.0
    }
  ]
}''')
print("Job:", JOB_SPEC["job_id"], "| Engine:", JOB_SPEC["engine"], "| Scenes:", len(JOB_SPEC["scenes"]))


In [ ]:
# Cell 3: Load GPU Pipeline & Generate All Scenes
import torch
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.enable_model_cpu_offload()
NUM_STEPS = 4
GUIDANCE = 0.0
W, H = (768, 1344) if JOB_SPEC.get("aspect_ratio", "9:16") == "9:16" else (1344, 768)

CHAR_LOCK = JOB_SPEC.get("character_lock", "")
NEG = JOB_SPEC.get("negative_prompt", "text, watermark, signature, blurry, deformed, extra limbs, 3d render")
BASE_SEED = int(JOB_SPEC.get("seed", 42))
manifest = {"job_id": JOB_SPEC["job_id"], "engine": JOB_SPEC["engine"], "files": []}

for idx, sc in enumerate(JOB_SPEC["scenes"]):
    sid = sc.get("scene_id") or f"scene_{idx + 1:02d}"
    full_prompt = f"{CHAR_LOCK} {sc['prompt']}".strip()
    gen = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(BASE_SEED + idx)
    t0 = time.time()
    img = pipe(prompt=full_prompt, negative_prompt=NEG, width=W, height=H, num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE, generator=gen).images[0]
    out_path = OUT_DIR / f"{sid}.png"
    img.save(out_path)
    print(f"[OK] {sid} -> {out_path.name} ({time.time() - t0:.1f}s)")
    manifest["files"].append(out_path.name)

(OUT_DIR / "output_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
zip_path = OUT_DIR.parent / f"ep01_30_days_funded_survival_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        zf.write(p, p.name)
print("PACKED:", zip_path, "| Total files:", len(manifest["files"]))


In [ ]:
# Cell 4: Optional Google Drive Auto-Backup on Colab
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drv_dir = pathlib.Path("/content/drive/MyDrive/AI_MONEY_SUITE_OUTPUTS/ep01_30_days_funded_survival")
        drv_dir.mkdir(parents=True, exist_ok=True)
        import shutil
        for p in sorted(OUT_DIR.glob("*")):
            shutil.copy2(p, drv_dir / p.name)
        print("SYNCED TO GOOGLE DRIVE:", drv_dir)
    except Exception as exc:
        print("Drive mount skipped (download /content/ep01_30_days_funded_survival_outputs.zip manually):", exc)
